# Capítulo 6 — La cuenta del taxímetro

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

> 🎬 **Escena.** Finanzas pregunta por qué la factura de Bedrock se ha triplicado en un mes. Nadie ha añadido clientes; hemos añadido memoria, herramientas e historial.

| | |
|---|---|
| **🧠 Componente agéntico** | Contexto: qué ve el modelo en cada llamada |
| **La necesidad** | Controlar qué entra en la ventana de contexto: menos coste, menos latencia y respuestas más centradas |
| **Lo que construimos** | Prompt con prefijo estable (*prompt caching*), contexto dinámico separado y ventana deslizante del historial |
| **🏠 En local** | ✅ Funciona igual en local y en la nube: es diseño del prompt y del historial. |
| **☁️ Por qué AWS al salir a Internet** | La API del modelo no guarda estado: **cada llamada reenvía todo**. En una conversación de 20 turnos se paga muchas veces el mismo contenido. En producción se mide (*tokens* por conversación), se cachea el prefijo y se limita el historial. |
| **En el ordenador sería…** | La **gestión de memoria del sistema operativo**: decidir qué se carga en la RAM, en qué orden y qué se descarga |
| **Requisitos del caso** | Eficiencia y coste (D6) |
| **Prerrequisito · duración** | Capítulo 5 · 45 min |
| **⚠️ Dónde se rompe la analogía** | Un sistema operativo pagina la memoria sin perder datos; **recortar el contexto sí pierde información**. Hay que elegir qué se olvida. |

In [ ]:
# Configuración común: ejecuta siempre esta celda primero
import os
from lab_utils import *

os.environ.setdefault("AWS_REGION", region())
cuaderno()

> 🎮 **¿Llegas tarde o algo se rompió?** La celda siguiente comprueba que tienes todo lo necesario. Si no, descomenta `ponerse_al_dia(6)`: carga la partida guardada al final del capítulo anterior o, si no existe, escribe el código de referencia y te dice qué recursos faltan y en qué capítulo se crean.

In [ ]:
requisitos(6)
# ponerse_al_dia(6)   # ← descomenta si llegas tarde o algo se rompió

## 🧠 La teoría: ingeniería de contexto

- Decidir **qué** entra en la ventana de contexto, **en qué orden** y **con qué límites**.
- **Coste y latencia** crecen con el contexto, porque se reenvía entero en cada llamada.
- **Prefijo estable** primero (se puede cachear); **contenido dinámico** al final; **historial** con ventana o resumen.
- Más contexto no es mejor: lo irrelevante **diluye la atención** del modelo.

## 1. El histórico del taxímetro

In [ ]:
taximetro_historico()

## 2. Experimento: una conversación larga

Ocho turnos en la **misma** sesión. Fíjate en cómo crecen los *tokens* de entrada **por turno**: cada mensaje reenvía el historial completo.

In [ ]:
CONVERSACION = ["Hola, soy Laura", "¿Dónde está mi pedido TN-1001?", "¿Y quién lo lleva?",
                "¿Qué artículos tiene el TN-1003?", "Quiero devolver la cafetera", "¿Cuánto tardará el reembolso?",
                "¿Me avisáis por WhatsApp?", "Gracias, eso es todo"]
sid_antes = new_session_id()
for mensaje in CONVERSACION:
    invocar(mensaje, session_id=sid_antes)

El coste por turno crece con la longitud de la conversación, así que el coste **total** crece aproximadamente con su cuadrado.

## 3. Las capas del contexto

| Capa | Contenido | Estabilidad | Origen |
|---|---|---|---|
| 1. Identidad y reglas | Rol, tono, art. 50, límites, formato | Estable → **cacheable** | *System prompt* |
| 2. Contexto de sesión | Fecha, canal, idioma | Por invocación | Código |
| 3. Memoria | Preferencias y hechos | Por cliente | Memory |
| 4. Conocimiento | Políticas | Por consulta | RAG (capítulo 8) |
| 5. Resultados de herramientas | Pedidos, envíos | Por turno | Herramientas |
| 6. Historial | Turnos previos | **Creciente** | *Session manager* + ventana |

## 4. Prompt con prefijo estable, contexto dinámico y ventana deslizante

In [ ]:
%%writefile NovaAssist/app/NovaAgent/prompts.py
# Prefijo ESTABLE (cacheable). El contexto dinámico se añade después, en main.py.
SYSTEM_PROMPT = """<identidad>
Eres Nova Assist, el asistente de postventa de TiendaNova. En tu primer mensaje te presentas como asistente de IA.
</identidad>

<reglas>
1. Nunca inventes estados, fechas ni importes: usa siempre las herramientas.
2. Los importes se expresan en euros al cliente, pero las herramientas trabajan en céntimos.
3. Para dudas de política (devoluciones, garantías, envíos) consulta la base de conocimiento y cita el documento.
4. Si una herramienta es denegada por política o el caso supera tus límites, usa escalar_a_humano con un resumen claro.
5. No pidas ni repitas datos de tarjetas de pago.
</reglas>

<formato>
Respuestas breves (máximo 5 frases), con el siguiente paso claro para el cliente.
</formato>"""

In [ ]:
%%writefile NovaAssist/app/NovaAgent/contexto.py
"""Contexto dinámico por invocación (va DESPUÉS del prefijo estable del system prompt)."""
from datetime import date


def contexto_sesion(canal: str, idioma: str) -> str:
    return (
        "<contexto_sesion>\n"
        f"fecha_actual: {date.today().isoformat()}\n"
        f"canal: {canal}\n"
        f"idioma_preferido: {idioma}\n"
        "</contexto_sesion>"
    )

In [ ]:
%%writefile NovaAssist/app/NovaAgent/main.py
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent
from strands.agent.conversation_manager import SlidingWindowConversationManager
from strands.models import BedrockModel

from contexto import contexto_sesion
from herramientas import HERRAMIENTAS
from memoria import crear_session_manager, retrieval_config
from prompts import SYSTEM_PROMPT
from settings import MODEL_ID, REGION

app = BedrockAgentCoreApp()


def metricas(result):
    """Herramientas usadas y consumo de tokens (para la verificación en los notebooks)."""
    try:
        m = result.metrics
        return sorted(m.tool_metrics.keys()), dict(m.accumulated_usage)
    except Exception:  # noqa: BLE001
        return [], {}


@app.entrypoint
def invoke(payload, context):
    actor_id = payload.get("cliente_id", "anonimo")  # ⚠️ provisional: en el capítulo 9 saldrá del token
    prompt_sistema = SYSTEM_PROMPT + "\n\n" + contexto_sesion(payload.get("canal", "web"), payload.get("idioma", "es"))
    with crear_session_manager(actor_id, context.session_id, retrieval_config()) as sm:
        agent = Agent(
            model=BedrockModel(model_id=MODEL_ID, region_name=REGION, temperature=0.2),
            system_prompt=prompt_sistema,
            tools=HERRAMIENTAS,
            session_manager=sm,
            conversation_manager=SlidingWindowConversationManager(window_size=20),
            callback_handler=None,
        )
        result = agent(payload.get("prompt", ""))
    herramientas, uso = metricas(result)
    return {"respuesta": str(result), "herramientas": herramientas, "tokens": uso}


if __name__ == "__main__":
    app.run()

In [ ]:
!cd "{PROJECT}" && agentcore deploy --yes

## 5. La misma conversación, con la ventana deslizante

In [ ]:
sid_despues = new_session_id()
for mensaje in CONVERSACION:
    invocar(mensaje, session_id=sid_despues)
taximetro()

Compara las dos sesiones en el resumen del taxímetro.

**Para seguir bajando el coste:**
- ***Prompt caching*:** el prefijo estable ya va primero. Activar el cacheo en el modelo (consulta la guía de *prompt caching* de Bedrock y Strands) hace que las lecturas de caché se cobren a una fracción del precio.
- **Resultados de herramienta compactos:** solo los campos necesarios.
- **`top_k` pequeño** en memoria y RAG.
- **Búsqueda semántica de herramientas** en el Gateway, en lugar de cargar todas (capítulo 7).

**Regla de diseño:** el plazo de devolución no se calcula en el prompt: lo calculará el código de la herramienta (capítulo 7). El modelo decide *qué* hacer; el código decide *si es válido*.

## Para pensar

- ¿Qué información sobra hoy en tu contexto?
- ¿Por qué el contenido dinámico va después del prefijo estable?

## 🔎 Lo que acabamos de descubrir

Hemos puesto orden en la RAM. Pero TiendaNova lanza una **app para repartidores** que necesita consultar envíos, y las herramientas están **pegadas al código de Nova**, con los datos dentro. ¿Copiamos todo en otro agente?

In [ ]:
deuda("D6", "parcial", 6, "ventana deslizante y prefijo cacheable; seguimiento en cap. 12")
cuaderno()
taximetro_cerrar(6)
guardar_partida(6)

---
➡️ **Siguiente capítulo:** `07_gateway.ipynb` — *Herramientas compartidas y gobernadas*